# 교시별 실습 Notebook

d2-p01-02

수업 페이지의 순서대로 실행합니다. 정답 블록과 선택 웹 자동화는 설명을 읽고 별도로 실행하세요.

In [ ]:
from pathlib import Path
import os

candidates = [Path.cwd(), *Path.cwd().parents]
course_root = next((p for p in candidates if (p / "raw/기존강사 수업자료").is_dir()), None)
if course_root is None:
    raise FileNotFoundError("수업 자료를 풀고 raw 폴더가 있는 위치에서 Notebook을 여세요.")
os.chdir(course_root)
print("수업 자료 폴더:", Path.cwd())

# 요약

Titanic 자료로 성별 생존율 막대그래프, 나이 히스토그램, 상관계수 히트맵을 만들고 저장합니다.

## 2일차 1~2교시 — 그래프의 구조와 세 가지 그림

2026년 10월 7일(수) · 2일차 1~2교시.

## 시작 전에

새 Notebook을 만들고 `day2_visualization`으로 이름을 바꿉니다. 1일차에 사용한 원본 `titanic.xlsx`를 그대로 읽습니다. pandas, openpyxl, matplotlib, seaborn이 필요합니다. 원본을 변경하지 않습니다.

코드 블록 하나를 Notebook의 코드 셀 하나에 붙여 넣고 위에서 아래로 Shift+Enter로 실행합니다. 앞 셀을 실행하지 않고 뒤 셀부터 실행하면 변수를 찾지 못할 수 있습니다. 파일 경로는 자료 폴더의 최상위 위치(`raw`가 있는 폴더)를 기준으로 합니다. 현재 위치가 다르면 파일 경로를 실제 파일 위치로 바꿉니다.

## 교시별 진행

- 1교시: 전날 생존율 표 복습 → Figure와 Axes 구분 → 성별 생존율 표 → 막대그래프 작성.
- 2교시: 나이 결측치를 제외한 히스토그램 → 상관계수 히트맵 → 이미지 파일 저장 → 그림의 단위 확인.

## 1. matplotlib, seaborn 기본 시각화 이해

Matplotlib의 Figure는 전체 그림, Axes는 축과 그래프가 있는 영역입니다. Seaborn은 DataFrame의 열 이름으로 통계 그래프를 만들 수 있습니다. 먼저 데이터와 그래프 크기를 준비합니다.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

titanic = pd.read_excel(Path("raw/기존강사 수업자료/titanic.xlsx"))
plots = Path("practice/plots")
plots.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot([1, 2, 3], [2, 4, 3], marker="o")
ax.set(title="Plot anatomy", xlabel="Step", ylabel="Value")
fig.tight_layout()
plt.show()
plt.close(fig)

Figure 안에 Axes를 만들고 제목·축·크기를 지정했습니다. 실제 데이터 그래프에도 이 구조를 사용합니다. Notebook에서 `plt.show()`는 그림을 보여주고 `plt.close(fig)`는 사용이 끝난 그림을 정리합니다.

**직접 실습:** 그림 크기를 `(9, 3)`으로 바꾸고 축 제목과 점 3개를 확인합니다.

## 2. 막대그래프, 히스토그램, 히트맵 생성

막대그래프는 범주별 값을 비교하고, 히스토그램은 숫자 값의 분포를 보여줍니다. 히트맵은 표의 값들을 색으로 표현합니다. 그리기 전에 계산한 표가 질문에 맞는지 먼저 봅니다.

### 성별 생존율 막대그래프

In [ ]:
sex_rates = titanic.groupby("sex")["survived"].agg(["count", "mean"])
print(sex_rates.round(4))
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(sex_rates.index, sex_rates["mean"] * 100)
ax.bar_label(bars, fmt="%.1f%%")
ax.set(title="Survival rate by sex", xlabel="Sex", ylabel="Survival rate (%)", ylim=(0, 100))
fig.tight_layout()
fig.savefig(plots / "survival_by_sex.png", dpi=150)
plt.show()
plt.close(fig)

여성 약 72.7%, 남성 약 19.1%의 막대가 나타납니다. 이 그림의 y축은 생존자 수가 아닌 비율입니다. 여성 466명, 남성 843명이라는 분모는 위 표에서 함께 확인합니다.

### 나이 분포 히스토그램

In [ ]:
known_age = titanic.dropna(subset=["age"])
print("그림에 포함된 승객:", len(known_age), "나이 미상:", titanic["age"].isna().sum())
fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(data=known_age, x="age", bins=20, ax=ax)
ax.set(title="Age distribution (known ages only)", xlabel="Age", ylabel="Passenger count")
fig.tight_layout()
fig.savefig(plots / "age_histogram.png", dpi=150)
plt.show()
plt.close(fig)

1,046명의 나이 분포입니다. 결측 263명은 0세로 그리지 않습니다. `bins`가 바뀌면 같은 데이터도 다르게 보일 수 있으므로 구간 수를 기록합니다.

### 수치 변수 상관계수 히트맵

In [ ]:
columns = ["pclass", "age", "sibsp", "parch", "fare", "survived"]
corr = titanic[columns].corr()
print(corr.round(2))
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, center=0, ax=ax)
ax.set_title("Numeric correlations")
fig.tight_layout()
fig.savefig(plots / "correlation_heatmap.png", dpi=150)
plt.show()
plt.close(fig)

대각선은 1이고 양수·음수 관계가 서로 다른 색으로 표시됩니다. 결측치를 제외하는 과정에서 변수 쌍마다 사용한 행 수가 다를 수 있습니다. 상관계수는 원인 관계를 입증하지 않으며, 성별처럼 문자열인 변수는 이 표에서 빠져 있습니다. pclass는 등급을 숫자로 표시했으므로 숫자 상관계수의 해석에도 주의합니다.

**직접 실습:** 성별 대신 객실 등급별 생존율을 그립니다. 히스토그램의 bins를 10과 30으로 바꾸고 분포 설명에서 달라지는 부분을 기록합니다.

## 직접 해보기와 정답

1. `bins=20`을 10으로 바꿔 실행한 뒤 30으로도 실행합니다. 나이 분포가 어떤 부분에서 달라 보이는지 두 문장으로 기록합니다.
2. 객실 등급별 생존율 표와 막대그래프를 만듭니다. y축은 0~100%로 고정합니다.

<details>
<summary>실습 정답 예시</summary>

```python
class_rates = titanic.groupby("pclass")["survived"].agg(["count", "mean"])
print(class_rates.round(4))
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(class_rates.index.astype(str), class_rates["mean"] * 100)
ax.bar_label(bars, fmt="%.1f%%")
ax.set(title="Survival by class", xlabel="Passenger class", ylabel="Survival rate (%)", ylim=(0, 100))
fig.tight_layout()
fig.savefig(plots / "survival_by_class.png", dpi=150)
plt.show()
plt.close(fig)
```

1등급부터 3등급까지 세 막대가 나타납니다. 등급의 인원 차이는 count 열에서 확인합니다. bins가 작으면 넓은 구간을 합쳐 큰 흐름이 보이고, bins가 크면 세부 모양이 보이지만 작은 변동도 강조됩니다. 원본 데이터가 바뀐 것은 아닙니다.

</details>

## 흔한 오류와 해결

| 증상 | 해결 |
| --- | --- |
| FileNotFoundError | 현재 폴더와 원본 위치를 확인하고 경로를 수정합니다. |
| 그림에 빈 축만 보임 | `ax.bar`·`sns.histplot` 셀을 실행했는지 확인합니다. |
| 한글 글꼴 경고 | 축·제목을 영어로 유지하거나 아래 안내로 설치된 한글 글꼴을 선택합니다. |
| 출력 그림과 저장 파일이 다름 | 그리기 → tight_layout → savefig → show → close 순서로 실행합니다. |

## 완료 체크

- [ ] 수업 폴더에서 uv로 Notebook을 열고 Kernel의 `sys.executable`이 수업 `.venv`인지 확인했습니다. [환경 안내](surl:14892)
- [ ] 실습을 저장한 뒤 Kernel을 재시작하고 선행 페이지의 준비 셀부터 위에서 아래로 다시 실행했습니다. 웹 수집은 저장한 자료와 성공·실패 기록을 구분했습니다.

- [ ] 세 그림을 Notebook에서 확인했습니다.
- [ ] `practice/plots`에서 저장한 PNG 파일을 열었습니다.
- [ ] 생존자 수와 생존율을 구분했습니다.
- [ ] 나이 미상 263명을 0세로 처리하지 않았습니다.

## 한글 그래프를 쓸 때: 설치된 글꼴 확인

예제의 영어 제목·축 이름은 Windows와 Mac에서 그대로 사용합니다. 한글로 바꾸고 싶다면 `sns.set_theme(...)`를 실행한 **뒤**, 그래프를 그리기 전에 아래 선택 코드를 실행합니다. Windows의 `Malgun Gothic`, Mac의 `AppleGothic` 등을 실제 설치 목록에서 찾습니다.

<details>
<summary>한글 폰트 선택 코드</summary>

```python
import platform
import matplotlib.pyplot as plt
from matplotlib import font_manager

installed = {font.name for font in font_manager.fontManager.ttflist}
candidates = (["AppleGothic", "Apple SD Gothic Neo", "NanumGothic"]
              if platform.system() == "Darwin"
              else ["Malgun Gothic", "NanumGothic"])
selected = next((name for name in candidates if name in installed), None)
if selected:
    plt.rcParams["font.family"] = selected
    plt.rcParams["axes.unicode_minus"] = False
    print("사용 글꼴:", selected)
else:
    print("설치된 한글 글꼴이 없습니다. 제목·축 이름을 영어로 유지하세요.")
```

Mac은 Font Book에서 글꼴 설치·활성화 상태를 확인할 수 있습니다. 설치 후 Kernel을 재시작하고 목록을 다시 확인합니다. [Matplotlib 글꼴 관리](https://matplotlib.org/stable/api/font_manager_api.html) · [Apple 글꼴 관리](https://support.apple.com/guide/font-book/install-and-validate-fonts-fntbk1000/mac)

</details>

## 파일 위치 확인

자료 폴더는 Windows `C:\kpc-finance-data`, Mac `~/kpc-finance-data`를 사용합니다. `Path.cwd()`가 `raw`를 포함한 최상위 폴더인지 확인합니다. 배포 Notebook의 준비 셀은 이 위치를 찾습니다. 파일명·공백·대소문자와 예제의 CSV 인코딩을 그대로 맞춥니다. [경로와 조작 안내](surl:14881)

## 실행 결과 확인

제공 파일로 실행한 행 수를 기준으로 설명합니다. 라이브 수집 자료나 라이브러리 버전이 달라지면 값이 달라질 수 있습니다.

<!-- period-navigation:start -->
## 교시별 이동

[교시별 전체 안내](surl:14891) · [공식 단원별 안내](surl:14880) · [이전: 10월 6일 7~8교시 — Titanic 탐색과 생존율](surl:14896) · [다음: 10월 7일 3교시 — 그래프에서 패턴과 인사이트 읽기](surl:14898)
<!-- period-navigation:end -->